# NFL Play Video Viewer

In [2]:
import re, json, requests, urllib.parse, asyncio, threading
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from playwright.async_api import async_playwright

HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120 Safari/537.36"}

_nfl_token = {"value": None}

async def _fetch_nfl_token() -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        holder = {}
        async def on_response(response):
            if "identity" in response.url and "token" in response.url:
                try:
                    body = await response.json()
                    if "accessToken" in body:
                        holder["token"] = body["accessToken"]
                except Exception:
                    pass
        page.on("response", on_response)
        await page.goto("https://www.nfl.com/videos/", wait_until="domcontentloaded", timeout=30000)
        await page.wait_for_timeout(4000)
        await browser.close()
    return holder.get("token")

def _run_in_thread(coro):
    """Run an async coroutine in a fresh thread with its own event loop."""
    result = {}
    def run():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result["value"] = loop.run_until_complete(coro)
        finally:
            loop.close()
    t = threading.Thread(target=run)
    t.start()
    t.join()
    return result.get("value")

def get_nfl_token(force_refresh=False) -> str:
    if _nfl_token["value"] and not force_refresh:
        return _nfl_token["value"]
    print("Getting NFL auth token (headless browser)...")
    token = _run_in_thread(_fetch_nfl_token())
    if not token:
        raise RuntimeError("Could not obtain NFL auth token")
    _nfl_token["value"] = token
    return token

In [3]:
def team_slug(display_name: str) -> str:
    """'Seattle Seahawks' -> 'seattle-seahawks', 'San Francisco 49ers' -> 'san-francisco-49ers'"""
    return display_name.lower().replace(" ", "-")

def get_games(season: int, season_type: int, week: int) -> list[dict]:
    """season_type: 1=preseason, 2=regular, 3=postseason"""
    url = (
        f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
        f"?seasontype={season_type}&week={week}&dates={season}"
    )
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    events = resp.json().get("events", [])
    stype_slug = {1: "pre", 2: "reg", 3: "post"}[season_type]
    games = []
    for e in events:
        comp = e["competitions"][0]
        home = next(t for t in comp["competitors"] if t["homeAway"] == "home")["team"]["displayName"]
        away = next(t for t in comp["competitors"] if t["homeAway"] == "away")["team"]["displayName"]
        away_slug = team_slug(away)
        home_slug = team_slug(home)
        nfl_game_slug = f"{away_slug}-at-{home_slug}-{season}-{stype_slug}-{week}"
        games.append({
            "label": f"{away} at {home}",
            "slug": nfl_game_slug,
            "away": away_slug,
            "home": home_slug,
            "week": week,
            "type": stype_slug,
            "season": season,
        })
    return games

def get_plays(game: dict) -> list[dict]:
    """
    Fetch all video clips for a game via the NFL content API.
    Falls back to the highlights page playlist if the API returns nothing.
    """
    game_tag = game["slug"]
    token = get_nfl_token()

    def _fetch(tok):
        return requests.get(
            f"https://api.nfl.com/content/v1/videos?tag={game_tag}&limit=200",
            headers={"Authorization": f"Bearer {tok}", "User-Agent": HEADERS["User-Agent"]}
        )

    resp = _fetch(token)
    if resp.status_code == 401:
        resp = _fetch(get_nfl_token(force_refresh=True))
    resp.raise_for_status()
    items = resp.json().get("items", [])

    if items:
        return [
            {
                "title": item["title"],
                "slug": item.get("slug", ""),
                "mcpID": item.get("mcpPlaybackId"),
            }
            for item in items
        ]

    # Fallback: highlights page playlist
    away, home, week, stype = game["away"], game["home"], game["week"], game["type"]
    if stype == "reg":
        hl_slug = f"{away}-vs-{home}-highlights-week-{week}"
    elif stype == "pre":
        hl_slug = f"{away}-vs-{home}-highlights-preseason-week-{week}"
    else:
        hl_slug = f"{away}-vs-{home}-highlights"
    hl_resp = requests.get(f"https://www.nfl.com/videos/{hl_slug}", headers=HEADERS)
    for s in re.findall(r"<script[^>]*>(.*?)</script>", hl_resp.text, re.DOTALL):
        if "playlist" in s and "mcpID" in s:
            try:
                pl = json.loads(s).get("playlist", [])
                filtered = [p for p in pl if any(t.get("slug") == game_tag for t in p.get("tags", []))]
                return filtered if filtered else pl
            except Exception:
                pass
    return []

def get_stream_url(clip: dict) -> str:
    """Get a signed HLS stream URL for a clip."""
    mcp_id = clip.get("mcpID")
    slug = clip.get("slug", "")

    # Fallback: scrape mcpID from the clip page
    if not mcp_id and slug:
        resp = requests.get(f"https://www.nfl.com/videos/{slug}", headers=HEADERS)
        for s in re.findall(r"<script[^>]*>(.*?)</script>", resp.text, re.DOTALL):
            if "mcpID" in s and "playlist" in s:
                try:
                    pl = json.loads(s).get("playlist", [])
                    if pl:
                        mcp_id = str(pl[0].get("mcpID", ""))
                        break
                except Exception:
                    pass
    if not mcp_id:
        raise RuntimeError(f"Could not find mcpID for: {slug}")

    token = get_nfl_token()

    def _call(tok):
        return requests.post(
            f"https://api.nfl.com/play/v1/asset/{mcp_id}",
            json={"asset": {"mcpID": str(mcp_id), "slug": slug, "videoSource": "nfl.com"},
                  "autoplay": True, "init": True, "live": False},
            headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json",
                     "Accept": "*/*", "Referer": "https://www.nfl.com/"}
        )

    resp = _call(token)
    if resp.status_code == 401:
        resp = _call(get_nfl_token(force_refresh=True))
    resp.raise_for_status()
    url = resp.json().get("accessUrl")
    if not url:
        raise RuntimeError(f"No accessUrl in response")
    return url

In [4]:
import datetime
current_year = datetime.datetime.now().year

season_sel = widgets.BoundedIntText(value=2025, min=2010, max=current_year, description="Season:", layout=widgets.Layout(width="200px"))
type_sel = widgets.Dropdown(options=[("Regular Season", 2), ("Postseason", 3), ("Preseason", 1)], description="Type:", layout=widgets.Layout(width="250px"))
week_sel = widgets.BoundedIntText(value=1, min=1, max=22, description="Week:", layout=widgets.Layout(width="200px"))
load_games_btn = widgets.Button(description="Load Games", button_style="info")
game_dropdown  = widgets.Dropdown(description="Game:", layout=widgets.Layout(width="70%"))
play_dropdown  = widgets.Dropdown(description="Play:", layout=widgets.Layout(width="70%"))
get_video_btn  = widgets.Button(description="Get Video", button_style="success")
status    = widgets.Output()
video_out = widgets.Output()

_plays_by_slug = {}

def on_load_games(b):
    with status:
        clear_output(wait=True)
        print(f"Loading week {week_sel.value} games for {season_sel.value}...")
    try:
        games = get_games(season_sel.value, type_sel.value, week_sel.value)
        game_dropdown.options = [(g["label"], g) for g in games]
        play_dropdown.options = []
        video_out.clear_output()
        with status:
            clear_output(wait=True)
            print(f"✓ {len(games)} games loaded. Select a game.")
    except Exception as e:
        with status:
            clear_output(wait=True)
            print(f"Error: {e}")

def on_game_change(change):
    game = change["new"]
    if not game:
        return
    with status:
        clear_output(wait=True)
        print(f"Loading clips for {game['label']}...")
    try:
        plays = get_plays(game)
        _plays_by_slug.clear()
        for p in plays:
            _plays_by_slug[p["slug"]] = p
        play_dropdown.options = [(p["title"], p["slug"]) for p in plays]
        with status:
            clear_output(wait=True)
            if len(plays) <= 1:
                print("✓ 1 clip found (full highlights only — individual play clips aren't available for this game).")
            else:
                print(f"✓ {len(plays)} clips loaded. Select a play and click Get Video.")
    except Exception as e:
        with status:
            clear_output(wait=True)
            print(f"Error: {e}")

def on_get_video(b):
    slug = play_dropdown.value
    title = dict(play_dropdown.options).get(slug, slug)
    clip = _plays_by_slug.get(slug)
    if not clip:
        return
    with video_out:
        clear_output(wait=True)
        print(f"Fetching stream for: {title}...")
    try:
        url = get_stream_url(clip)
        with video_out:
            clear_output(wait=True)
            display(HTML(f"""
                <b>{title}</b><br><br>
                <video controls width="800">
                  <source src="{url}" type="application/x-mpegURL">
                </video>
                <br><small><a href="{url}" target="_blank">Open stream URL</a></small>
            """))
    except Exception as e:
        with video_out:
            clear_output(wait=True)
            print(f"Error: {e}")

load_games_btn.on_click(on_load_games)
game_dropdown.observe(on_game_change, names="value")
get_video_btn.on_click(on_get_video)

display(
    widgets.HBox([season_sel, type_sel, week_sel, load_games_btn]),
    game_dropdown,
    play_dropdown,
    get_video_btn,
    status,
    video_out,
)

Dropdown(description='Game:', layout=Layout(width='70%'), options=(), value=None)

Dropdown(description='Play:', layout=Layout(width='70%'), options=(), value=None)

Button(button_style='success', description='Get Video', style=ButtonStyle())

Output()

Output()

---
## QB Throws Browser
Enter a QB name and season to browse all their passing clips across every week.

In [5]:
def get_team_for_player(player_slug: str, season: int, token: str) -> str | None:
    """Find the team slug for a player by scanning a sample of their recent clips."""
    r = requests.get(
        f"https://api.nfl.com/content/v1/videos?tag={player_slug}&limit=10",
        headers={"Authorization": f"Bearer {token}", "User-Agent": HEADERS["User-Agent"]}
    )
    if r.status_code != 200:
        return None
    for item in r.json().get("items", []):
        for tag in item.get("tags", []):
            if tag.get("externalSourceName") == "teams":
                return tag["slug"]  # e.g. "san-francisco-49ers"
    return None

def get_qb_clips(player_name: str, season: int, progress_out=None) -> list[dict]:
    """
    Fetch all passing clips for a QB in a given season.
    Strategy: iterate over all weeks of the season, get game clips for the QB's team,
    then filter to clips tagged with the QB player slug.
    """
    player_slug = player_name.lower().strip().replace(" ", "-")
    token = get_nfl_token()

    # Find which team the QB plays for
    team_slug = get_team_for_player(player_slug, season, token)
    if not team_slug:
        raise RuntimeError(f"Could not find team for '{player_name}'. Check name spelling.")

    def _log(msg):
        if progress_out:
            with progress_out:
                clear_output(wait=True)
                print(msg)
        else:
            print(msg)

    _log(f"Found team: {team_slug}. Scanning all 2025 weeks...")

    auth_headers = {"Authorization": f"Bearer {token}", "User-Agent": HEADERS["User-Agent"]}

    # Exclude non-pass clips
    exclude_keywords = ["sack", " rush", "rushing", " run ", "scramble", "best plays", "highlights"]

    all_clips = []
    seen_slugs = set()

    for season_type in [2, 3]:  # regular season then postseason
        max_week = 19 if season_type == 2 else 5
        for week in range(1, max_week):
            espn_url = (
                f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
                f"?seasontype={season_type}&week={week}&dates={season}"
            )
            espn_r = requests.get(espn_url, headers=HEADERS)
            if espn_r.status_code != 200:
                continue
            events = espn_r.json().get("events", [])
            if not events:
                break

            # Find the game the QB's team played in
            game_tag = None
            for e in events:
                comp = e["competitions"][0]
                home = next(t for t in comp["competitors"] if t["homeAway"] == "home")["team"]["displayName"]
                away = next(t for t in comp["competitors"] if t["homeAway"] == "away")["team"]["displayName"]
                home_s = home.lower().replace(" ", "-")
                away_s = away.lower().replace(" ", "-")
                if home_s == team_slug or away_s == team_slug:
                    stype = {2: "reg", 3: "post"}[season_type]
                    game_tag = f"{away_s}-at-{home_s}-{season}-{stype}-{week}"
                    break

            if not game_tag:
                continue

            _log(f"Scanning {game_tag}...")

            # Fetch all clips for this game
            game_r = requests.get(
                f"https://api.nfl.com/content/v1/videos?tag={game_tag}&limit=200",
                headers=auth_headers
            )
            if game_r.status_code != 200:
                continue
            items = game_r.json().get("items", [])

            # Filter to clips tagged with this QB
            for item in items:
                slug = item.get("slug", "")
                if slug in seen_slugs:
                    continue
                tags = item.get("tags", [])
                if not any(t.get("slug") == player_slug for t in tags):
                    continue
                title = item.get("title", "")
                title_lower = title.lower()
                if any(kw in title_lower for kw in exclude_keywords):
                    continue
                seen_slugs.add(slug)
                all_clips.append({
                    "title": title,
                    "slug": slug,
                    "mcpID": item.get("mcpPlaybackId"),
                    "week": f"{'REG' if season_type == 2 else 'POST'} Wk{week}",
                })

    return all_clips

# ── QB Browser UI ────────────────────────────────────────────────────
qb_name_input = widgets.Text(
    placeholder="e.g. Brock Purdy",
    description="QB Name:",
    layout=widgets.Layout(width="300px")
)
qb_season_sel = widgets.BoundedIntText(
    value=2025, min=2015, max=current_year,
    description="Season:",
    layout=widgets.Layout(width="180px")
)
load_qb_btn    = widgets.Button(description="Load QB Clips", button_style="warning")
qb_clip_dd     = widgets.Dropdown(description="Clip:", layout=widgets.Layout(width="80%"))
get_qb_vid_btn = widgets.Button(description="Get Video", button_style="success")
qb_status      = widgets.Output()
qb_video_out   = widgets.Output()

_qb_clips = {}

def on_load_qb(b):
    name = qb_name_input.value.strip()
    if not name:
        return
    qb_clip_dd.options = []
    qb_video_out.clear_output()
    with qb_status:
        clear_output(wait=True)
        print(f"Scanning {qb_season_sel.value} season for {name}... (this takes ~30s)")
    try:
        clips = get_qb_clips(name, qb_season_sel.value, progress_out=qb_status)
        _qb_clips.clear()
        for c in clips:
            _qb_clips[c["slug"]] = c
        qb_clip_dd.options = [(f"[{c['week']}] {c['title']}", c["slug"]) for c in clips]
        with qb_status:
            clear_output(wait=True)
            if not clips:
                print(f"No passing clips found for '{name}' in {qb_season_sel.value}.")
            else:
                print(f"✓ {len(clips)} clips loaded for {name} ({qb_season_sel.value} season).")
    except Exception as e:
        with qb_status:
            clear_output(wait=True)
            print(f"Error: {e}")

def on_get_qb_video(b):
    slug = qb_clip_dd.value
    title = dict(qb_clip_dd.options).get(slug, slug)
    clip = _qb_clips.get(slug)
    if not clip:
        return
    with qb_video_out:
        clear_output(wait=True)
        print(f"Fetching stream for: {title}...")
    try:
        url = get_stream_url(clip)
        with qb_video_out:
            clear_output(wait=True)
            display(HTML(f"""
                <b>{title}</b><br><br>
                <video controls width="800">
                  <source src="{url}" type="application/x-mpegURL">
                </video>
                <br><small><a href="{url}" target="_blank">Open stream URL</a></small>
            """))
    except Exception as e:
        with qb_video_out:
            clear_output(wait=True)
            print(f"Error: {e}")

load_qb_btn.on_click(on_load_qb)
get_qb_vid_btn.on_click(on_get_qb_video)

display(
    widgets.HBox([qb_name_input, qb_season_sel, load_qb_btn]),
    qb_clip_dd,
    get_qb_vid_btn,
    qb_status,
    qb_video_out,
)

Dropdown(description='Clip:', layout=Layout(width='80%'), options=(), value=None)

Button(button_style='success', description='Get Video', style=ButtonStyle())

Output()

Output()